# 🧮 Creating Text Embeddings

**Transform text into numerical vectors for semantic search**

---

## 📋 Overview

**What you'll learn:**
- What are embeddings and why they matter
- Create embeddings with sentence-transformers
- Choose the right embedding model
- Understand vector dimensions
- Batch processing for efficiency

**Time estimate:** ⏱️ 45 minutes | **Difficulty:** 🟡 Intermediate

---

## 🎯 Learning Objectives

1. ✅ Understand what embeddings are
2. ✅ Create embeddings from text
3. ✅ Compare different models
4. ✅ Optimize for performance
5. ✅ Handle edge cases

## 📖 What are Embeddings?

**Simple explanation:**
- Embeddings = Numbers that represent meaning
- Similar meanings = Similar numbers
- Enable computers to understand text

**Example:**
```
"dog" → [0.2, 0.8, 0.1, ...] (384 numbers)
"puppy" → [0.3, 0.7, 0.2, ...] (similar!)
"car" → [0.9, 0.1, 0.8, ...] (different!)
```

**Why use them?**
- Semantic search (find similar meaning)
- Recommendations (find related items)
- Clustering (group similar texts)
- Classification (categorize text)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List
import time

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"✅ Loaded model: {model.get_sentence_embedding_dimension()} dimensions")

## 🔨 Creating Your First Embeddings

In [ ]:
# Single text
text = "I love machine learning"
embedding = model.encode(text)

print(f"Text: {text}")
print(f"Embedding shape: {embedding.shape}")
print(f"First 5 values: {embedding[:5]}")
print(f"Vector range: [{embedding.min():.3f}, {embedding.max():.3f}]")

In [ ]:
# Multiple texts (batch processing - faster!)
texts = [
    "The quick brown fox jumps",
    "A fast auburn fox leaps",  # Similar meaning
    "I enjoy eating pizza",     # Different meaning
]

embeddings = model.encode(texts)

print(f"Created {len(embeddings)} embeddings")
print(f"Shape: {embeddings.shape}")
print(f"Memory: {embeddings.nbytes / 1024:.2f} KB")

## 📊 Comparing Embedding Models

In [ ]:
# Compare different models
models_to_test = [
    ('all-MiniLM-L6-v2', 'Fast, lightweight'),
    ('all-mpnet-base-v2', 'Best quality'),
]

test_text = "Machine learning is fascinating"

print("Model Comparison:\n" + "="*60)

for model_name, description in models_to_test:
    model_test = SentenceTransformer(model_name)
    
    start = time.time()
    emb = model_test.encode(test_text)
    elapsed = (time.time() - start) * 1000
    
    print(f"\n{model_name}:")
    print(f"  Description: {description}")
    print(f"  Dimensions: {len(emb)}")
    print(f"  Speed: {elapsed:.1f}ms")
    print(f"  Memory: {emb.nbytes / 1024:.2f} KB")

## ⚡ Performance Optimization

In [ ]:
class EmbeddingGenerator:
    """Efficient embedding generation with caching."""
    
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
        self.cache = {}
    
    def encode(self, texts: List[str], use_cache: bool = True) -> np.ndarray:
        """Encode texts with optional caching."""
        if use_cache:
            # Check cache
            uncached = [t for t in texts if t not in self.cache]
            
            if uncached:
                # Encode only uncached texts
                new_embeddings = self.model.encode(uncached)
                for text, emb in zip(uncached, new_embeddings):
                    self.cache[text] = emb
            
            # Return from cache
            return np.array([self.cache[t] for t in texts])
        else:
            return self.model.encode(texts)
    
    def get_cache_stats(self) -> dict:
        return {
            'cached_items': len(self.cache),
            'cache_size_kb': sum(v.nbytes for v in self.cache.values()) / 1024
        }

# Test caching
generator = EmbeddingGenerator()

texts = ["test"] * 100  # Same text repeated

# First call - no cache
start = time.time()
emb1 = generator.encode(texts, use_cache=False)
time_no_cache = time.time() - start

# Second call - with cache
start = time.time()
emb2 = generator.encode(texts, use_cache=True)
time_with_cache = time.time() - start

print(f"Without cache: {time_no_cache*1000:.1f}ms")
print(f"With cache: {time_with_cache*1000:.1f}ms")
print(f"Speedup: {time_no_cache/time_with_cache:.1f}x")
print(f"\nCache stats: {generator.get_cache_stats()}")

## ✅ Summary

### Key Takeaways:
- 🧮 **Embeddings**: Numbers representing meaning
- 📏 **Dimensions**: 384-768 typical
- ⚡ **Batch processing**: Much faster
- 💾 **Caching**: Avoid recomputation
- 🎯 **Model choice**: Balance speed vs quality

### Model Recommendations:
- **Fast**: all-MiniLM-L6-v2 (384 dims)
- **Quality**: all-mpnet-base-v2 (768 dims)
- **Multilingual**: paraphrase-multilingual-MiniLM-L12-v2

### Next: `04_embeddings_vectors/02_vector_similarity.ipynb`